# Wifaq 1448 — timestamp repair v2

Gemini writes the lessons' text well, but on long clips its timestamps drift: some transcripts run past the end of the recording (49:21 in a 33-minute lesson), others stop minutes early. A green time on a card must land on the right sentence, so this notebook corrects every timestamp **from the audio itself**:

1. Whisper (free, on Colab's GPU) listens to the recording and notes when each word is spoken. Its text is weaker than Gemini's, so only its timings are used.
2. Each Gemini line is found in Whisper's words and gets Whisper's time. Lines Whisper didn't catch are placed in proportion between their neighbours.
3. Stretches where the audio has plenty of speech but the transcript has almost no text are listed as **text missing**, so only those lessons need transcribing again.

**No Gemini requests are used.** It runs side by side with the transcription notebook: it only touches finished transcripts, never the inventory, the log or `_pipeline`, and it leaves alone any transcript the transcription notebook is still going to redo.

**What changes in Drive**
- Each corrected transcript gets new timestamps and a `- وقت:` line in its header (and `- خلا:` when text is missing). The text itself is not changed.
- An untouched copy of every transcript it changes goes to `transcripts/_original/`.
- `_whisper/` keeps Whisper's word timings, so a lesson is never listened to twice.
- `timestamp_fix.csv` (one row per lesson) and `timestamp_fix.md` (summary: lessons with text missing, lessons that couldn't be matched, biggest corrections). Claude reads these.

**v2 (3 Oct):** reads the audio with ffmpeg (the first run stopped on every lesson with a PyAV error), understands lines written as `01:03 [0:01:03] …`, and adds cell 6, which makes light audio copies for the card pages.

**How to run:** Runtime → **Change runtime type → T4 GPU** → Save. Then Runtime → **Run all**. Hidaya 4 goes first. A 45-minute lesson takes roughly 1–3 minutes; it stops by itself after `MAX_HOURS`, and the next run carries on. Colab gives free GPU time in limited amounts per day; if it says no GPU is available, try again later.

In [ ]:
# @title 1 · Connect Drive and install Whisper (2–3 minutes)
from google.colab import drive
drive.mount('/content/drive')
!pip -q install faster-whisper rapidfuzz
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — choose Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title 2 · Settings
DRIVE = "/content/drive/MyDrive"
OUT_DIR = f"{DRIVE}/Wifaq 1448"

# Same order as the transcription notebook (Hidaya 3 last).
BOOK_ORDER = [
    "ہدایہ جلد رابع",
    "مشکوۃ المصابیح جلد اول",
    "مشکوۃ المصابیح جلد دوم",
    "تیسیر مصطلح الحدیث",
    "شرح نخبۃ الفکر",
    "التبیان فی علوم القرآن",
    "بیضاوی",
    "آئینہ قادیانیت",
    "ہدایہ جلد ثالث",
]
ONLY_BOOKS = []          # e.g. ["ہدایہ جلد رابع"] to do just that book; empty = every book, in the order above

WHISPER_MODEL = "large-v3"   # most accurate; "large-v3-turbo" is about 3× faster with slightly weaker Urdu
MAX_HOURS = 4                # stops by itself after this long (Colab's free GPU time is limited)
REFIX = False                # True = check transcripts that were already corrected once more

# Transcripts with text missing: False = just list them in timestamp_fix.md (recommended at first).
# True = also move them to transcripts/_redo, so the transcription notebook transcribes them again.
MOVE_GAPS_FOR_REDO = False

In [ ]:
# @title 3 · Functions (run once, no need to edit)
# ------------------------------------------------------------------ timestamp repair: pure functions
# Gemini writes good text but its timestamps can drift on long clips. Whisper's text is weaker, but its
# timings come from the audio itself. We line Gemini's sentences up against Whisper's words and take
# the time from Whisper. No Gemini request is used.
import re, unicodedata, bisect
import numpy as np
from rapidfuzz import fuzz

# "[0:01:03] text", also "01:03 [0:01:03] text", "- [0:01:03] text", "**[0:01:03]** text" (some models add these)
TS_LINE = re.compile(r"^(\s*(?:[-*•]\s+)?)(?:\*\*)?(?:\(?\d{1,2}:\d{2}(?::\d{2})?\)?\s+)?"
                     r"\[(?:(\d{1,2}):)?(\d{1,4}):(\d{2})\](?:\*\*)?(.*)$")
_FOLD = str.maketrans({
    "ي": "ی", "ى": "ی", "ې": "ی", "ے": "ی", "ئ": "ی", "ك": "ک", "ه": "ہ", "ة": "ہ", "ۃ": "ہ", "ۀ": "ہ",
    "ۂ": "ہ", "ھ": "ہ", "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ؤ": "و", "ء": None, "ـ": None,
    "٠": None, "١": None, "٢": None, "٣": None, "٤": None, "٥": None, "٦": None, "٧": None, "٨": None, "٩": None,
    "۰": None, "۱": None, "۲": None, "۳": None, "۴": None, "۵": None, "۶": None, "۷": None, "۸": None, "۹": None,
})
_KEEP = re.compile(r"[^\w\s]|\d|_")


def norm_text(s):
    """Fold Arabic/Urdu spelling variants, drop diacritics, punctuation and digits."""
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("[؟]", " ").replace("ع:", " ")
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn").translate(_FOLD)
    s = _KEEP.sub(" ", s.lower())
    return re.sub(r"\s+", " ", s).strip()


def fmt_hms(sec):
    sec = int(round(max(sec, 0)))
    return f"{sec // 3600}:{(sec % 3600) // 60:02d}:{sec % 60:02d}"


def split_transcript(text):
    """-> (header, body_lines). Header is everything up to and including the '---' line."""
    lines = text.split("\n")
    for i, l in enumerate(lines[:80]):
        if l.strip() == "---":
            return "\n".join(lines[:i + 1]), lines[i + 1:]
    return "", lines


def parse_lines(body_lines):
    """Timestamped lines -> list of dicts with index into body_lines, old seconds, normalized text."""
    out = []
    for k, line in enumerate(body_lines):
        m = TS_LINE.match(line)
        if not m:
            continue
        sec = int(m.group(2) or 0) * 3600 + int(m.group(3)) * 60 + int(m.group(4))
        out.append({"k": k, "old": sec, "raw": m.group(5), "norm": norm_text(m.group(5))})
    return out


# ------------------------------------------------------------------ whisper side
class WhisperText:
    """Whisper words joined into one normalized string, with a time for every character."""

    def __init__(self, words):
        parts, times = [], []
        for start, end, w in words:
            t = norm_text(w)
            if not t:
                continue
            if parts:
                parts.append(" ")
                times.append(start)
            span = max(end - start, 0.01)
            for j, _ in enumerate(t):
                times.append(start + span * j / max(len(t), 1))
            parts.append(t)
        self.text = "".join(parts)
        self.times = np.array(times, dtype=float) if times else np.zeros(1)

    def time_at(self, i):
        if len(self.text) == 0:
            return 0.0
        i = int(min(max(round(i), 0), len(self.text) - 1))
        return float(self.times[i])

    def char_at_time(self, t):
        return int(np.searchsorted(self.times, t))


# ------------------------------------------------------------------ alignment
def _weighted_lis(anchors):
    """Keep the heaviest subset of anchors whose Whisper positions rise with the line order."""
    n = len(anchors)
    if n == 0:
        return []
    best = [a["score"] for a in anchors]
    prev = [-1] * n
    for i in range(n):
        for j in range(max(0, i - 60), i):           # neighbours are enough; keeps it fast
            if anchors[j]["w0"] < anchors[i]["w0"] and best[j] + anchors[i]["score"] > best[i]:
                best[i], prev[i] = best[j] + anchors[i]["score"], j
    i = int(np.argmax(best))
    keep = []
    while i != -1:
        keep.append(anchors[i])
        i = prev[i]
    return keep[::-1]


def align(lines, wt, duration, min_chars=10):
    """Find each Gemini line in Whisper's text. Returns (new_times, anchors, stats)."""
    W, n = wt.text, len(lines)
    if not lines or len(W) < 50:
        return None, [], {"reason": "Whisper heard almost nothing"}
    # Gemini character positions (cumulative), so a line between two anchors can be placed in proportion.
    cg, pos = [], 0
    for ln in lines:
        cg.append(pos)
        pos += len(ln["norm"]) + 1
    g_total = max(pos, 1)
    ratio = len(W) / g_total                          # Whisper chars per Gemini char, roughly 1
    def find(q, lo, hi, need):
        lo, hi = int(max(0, lo)), int(min(len(W), hi))
        if hi - lo < len(q):
            return None
        r = fuzz.partial_ratio_alignment(q, W[lo:hi], score_cutoff=min(need, 100))
        return None if r is None else (r.score, lo + r.dest_start, lo + r.dest_end)

    def need_for(q):
        return 64 if len(q) >= 45 else (72 if len(q) >= 25 else 82)

    def confirmed(i, w_end, g_end):
        """After a long jump, the next lines must also be found right after it."""
        hits = tries = 0
        for j in range(i + 1, n):
            q2 = lines[j]["norm"]
            if len(q2.replace(" ", "")) < 15:
                continue
            tries += 1
            e2 = w_end + (cg[j] - g_end) * ratio
            if find(q2, e2 - 1200, e2 + 1200 + len(q2), need_for(q2)):
                hits += 1
            if tries == 4:
                break
        return tries > 0 and hits >= min(2, tries)

    anchors = []
    last_w, last_g = 0, 0
    for i, ln in enumerate(lines):
        q = ln["norm"]
        if len(q.replace(" ", "")) < min_chars:
            continue
        need = need_for(q)
        expect = last_w + (cg[i] - last_g) * ratio
        half = max(1500, 0.6 * (cg[i] - last_g) * ratio, len(q) * 4)
        hit = find(q, min(expect - half, len(W) - len(q)), expect + half + len(q), need)
        if hit is None and len(q) >= 25:
            # Gemini may have skipped a stretch: look further ahead, but ask for a closer match
            # and check that the following lines carry on from there.
            far = find(q, expect + half, expect + 9000 + len(q), need + 6)
            if far and confirmed(i, far[2], cg[i] + len(q)):
                hit = far
        if hit is None:
            continue
        score, w0, w1 = hit
        anchors.append({"i": i, "w0": w0, "w1": w1, "score": score * min(len(q), 120) / 120})
        last_w, last_g = w1, cg[i] + len(q)
    eligible = sum(1 for ln in lines if len(ln["norm"].replace(" ", "")) >= min_chars)
    kept = _weighted_lis(anchors)
    stats = {"eligible": eligible, "found": len(anchors), "kept": len(kept),
             "coverage": round(len(kept) / max(eligible, 1), 3)}
    if len(kept) < max(8, 0.25 * eligible):
        stats["reason"] = f"only {len(kept)} of {eligible} lines could be matched"
        return None, kept, stats
    # place every line: anchors exactly, the rest in proportion between neighbouring anchors
    ka = [(a["i"], a["w0"]) for a in kept]
    idx = [a[0] for a in ka]
    new = []
    for i in range(n):
        j = bisect.bisect_left(idx, i)
        if j < len(idx) and idx[j] == i:
            w = ka[j][1]
        elif j == 0:                                   # before the first anchor
            i1, w1 = ka[0]
            w = w1 - (cg[i1] - cg[i]) * ratio
        elif j == len(idx):                            # after the last anchor
            i0, w0 = ka[-1]
            w = w0 + (cg[i] - cg[i0]) * ratio
        else:
            (i0, w0), (i1, w1) = ka[j - 1], ka[j]
            f = (cg[i] - cg[i0]) / max(cg[i1] - cg[i0], 1)
            w = w0 + f * (w1 - w0)
        new.append(min(max(wt.time_at(w), 0.0), duration))
    for i in range(1, n):                              # never go backwards
        new[i] = max(new[i], new[i - 1])
    shifts = [abs(new[i] - lines[i]["old"]) for i in range(n)]
    stats.update({"median_shift": round(float(np.median(shifts)), 1), "max_shift": round(float(max(shifts)), 1),
                  "end_old": lines[-1]["old"], "end_new": round(new[-1], 1)})
    return new, kept, stats


def find_gaps(lines, kept, wt, duration, min_sec=75):
    """Stretches where Whisper heard a lot of speech but Gemini wrote little: likely skipped text."""
    if not kept:
        return []
    cg, pos = [], 0
    for ln in lines:
        cg.append(pos)
        pos += len(ln["norm"]) + 1
    g_end = pos
    ratio = len(wt.text) / max(g_end, 1)
    pts = [(-1, 0, 0)] + [(a["i"], a["w0"], cg[a["i"]]) for a in kept] + [(len(lines), len(wt.text), g_end)]
    gaps = []
    for (ia, wa, ga), (ib, wb, gb) in zip(pts, pts[1:]):
        ta, tb = wt.time_at(wa), (wt.time_at(wb) if wb < len(wt.text) else duration)
        w_chars, g_chars = wb - wa, (gb - ga) * ratio
        if tb - ta >= min_sec and w_chars >= 700 and g_chars < 0.35 * w_chars:
            gaps.append((round(ta), round(tb)))
    return gaps


def render(header, body_lines, lines, new_times, note_lines):
    body = list(body_lines)
    for ln, t in zip(lines, new_times):
        m = TS_LINE.match(body[ln["k"]])
        body[ln["k"]] = f"{m.group(1)}[{fmt_hms(t)}]{m.group(5)}"
    head = header.split("\n")
    # put our notes just above the closing '---' (after an existing '- جانچ:' line, if any)
    while head and head[-1].strip() in ("", "---"):
        head.pop()
    head = [h for h in head if not h.startswith("- وقت:") and not h.startswith("- خلا:")]
    return "\n".join(head + note_lines + ["", "---"] + body)


# ------------------------------------------------------------------ timestamp repair: the run
import os, csv, json, glob, time, shutil, hashlib, subprocess, datetime
from zoneinfo import ZoneInfo

PK = ZoneInfo("Asia/Karachi")
TX_DIR = os.path.join(OUT_DIR, "transcripts")
ORIG_DIR = os.path.join(TX_DIR, "_original")     # untouched copies of every transcript we change
REDO_DIR = os.path.join(TX_DIR, "_redo")         # transcripts moved here are done again by the transcription notebook
WH_DIR = os.path.join(OUT_DIR, "_whisper")       # Whisper's word timings, kept so nothing is heard twice
LOG_PATH = os.path.join(OUT_DIR, "timestamp_fix.csv")
REPORT_PATH = os.path.join(OUT_DIR, "timestamp_fix.md")
LOCAL = "/content/tsfix" if os.path.isdir("/content") else "/tmp/tsfix"
BROKEN_MARK = "ٹرانسکرپٹ نہیں بن سکا"
HARD_FAILS = ("no timestamps", "empty answer", "blocked")


def now_pk():
    return datetime.datetime.now(PK)


def clock(dt):
    return dt.strftime("%I:%M %p").lstrip("0").lower()


def _norm_name(s):
    return norm_text(s).replace(" ", "")


def book_rank(book):
    nb = _norm_name(book)
    for i, b in enumerate(BOOK_ORDER):
        if _norm_name(b) and _norm_name(b) in nb:
            return i
    return len(BOOK_ORDER)


def header_fields(header):
    f = {}
    for line in header.split("\n"):
        if line.startswith("- ") and ":" in line:
            k, v = line[2:].split(":", 1)
            f[k.strip()] = v.strip()
    return f


def list_transcripts():
    rows = []
    for book_dir in sorted(glob.glob(os.path.join(TX_DIR, "*"))):
        book = os.path.basename(book_dir)
        if not os.path.isdir(book_dir) or book.startswith("_"):
            continue
        if ONLY_BOOKS and not any(_norm_name(b) in _norm_name(book) for b in ONLY_BOOKS):
            continue
        for path in glob.glob(os.path.join(book_dir, "*.md")):
            try:
                with open(path, encoding="utf-8") as f:
                    head = f.read(4000)
            except OSError:
                continue
            fields = header_fields(split_transcript(head)[0] or head)
            lesson = fields.get("سبق", "")
            rows.append({"path": path, "book": book, "file": os.path.basename(path),
                         "lesson": int(lesson) if lesson.isdigit() else 9999,
                         "audio": fields.get("راستہ", ""), "model": fields.get("ماڈل", ""),
                         "check": fields.get("جانچ", ""), "fixed": "وقت" in fields})
    rows.sort(key=lambda r: (book_rank(r["book"]), r["lesson"], r["file"]))
    return rows


def unfinished(text, check):
    """True when the transcription notebook will redo this transcript anyway."""
    if BROKEN_MARK in text or any(h in check for h in HARD_FAILS):
        return True
    for a, b in re.findall(r"(?<!nothing transcribed \()stopped at (\d+:\d\d) of (\d+:\d\d)", check):
        sec = lambda x: int(x.split(":")[0]) * 60 + int(x.split(":")[1])
        if sec(a) < 0.25 * sec(b):
            return True
    return False


# ------------------------------------------------------------------ Whisper
_PIPE = []


def _preload_cuda_libs():
    """Colab sometimes can't find cuDNN for faster-whisper; load the copies pip installed with torch."""
    import ctypes, site
    for sp in site.getsitepackages():
        for pat in ("nvidia/cublas/lib/libcublas*.so*", "nvidia/cudnn/lib/libcudnn*.so*"):
            for lib in sorted(glob.glob(os.path.join(sp, pat))):
                try:
                    ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass


def whisper_pipe():
    if _PIPE:
        return _PIPE[0]
    _preload_cuda_libs()
    import torch
    from faster_whisper import WhisperModel, BatchedInferencePipeline
    if torch.cuda.is_available():
        name, dev, ctype, batch = WHISPER_MODEL, "cuda", "float16", 16
    else:
        print("⚠ No GPU in this runtime (Runtime → Change runtime type → T4 GPU). "
              "Using the small model on the CPU — this is slow and less accurate.")
        name, dev, ctype, batch = "small", "cpu", "int8", 4
    print(f"Loading Whisper {name} on {dev}…", flush=True)
    model = WhisperModel(name, device=dev, compute_type=ctype)
    _PIPE.append((BatchedInferencePipeline(model=model), batch, name))
    return _PIPE[0]


def probe_seconds(path):
    try:
        out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                              "-of", "default=nw=1:nk=1", path], capture_output=True, text=True, timeout=300).stdout
        return float(out.strip())
    except Exception:
        return 0.0


def local_copy(src):
    """Copy the recording to the Colab disk first: reading straight from Drive can cut audio short."""
    os.makedirs(LOCAL, exist_ok=True)
    dst = os.path.join(LOCAL, "src_" + hashlib.md5(src.encode("utf-8")).hexdigest()[:12] + os.path.splitext(src)[1])
    want = os.path.getsize(src)
    for _ in range(3):
        if os.path.exists(dst) and os.path.getsize(dst) == want:
            return dst
        try:
            shutil.copyfile(src, dst + ".part")
            os.replace(dst + ".part", dst)
        except OSError as e:
            print(f"   … copying from Drive failed ({e}); trying again", flush=True)
            time.sleep(10)
    if os.path.exists(dst) and os.path.getsize(dst) == want:
        return dst
    raise RuntimeError("the recording could not be copied from Drive completely")


def load_audio(path, sr=16000):
    """Decode with ffmpeg (not PyAV: Colab's newer PyAV breaks faster-whisper's own decoder)."""
    out = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", path, "-vn", "-ac", "1", "-ar", str(sr),
                          "-f", "f32le", "-"], capture_output=True, check=True, timeout=3600).stdout
    return np.frombuffer(out, np.float32).copy()


def run_whisper(audio_path):
    """-> (duration_sec, [(start, end, word), ...], model_name)"""
    pipe, batch, name = whisper_pipe()
    audio = load_audio(audio_path)
    segs, info = pipe.transcribe(audio, language="ur", batch_size=batch, word_timestamps=True)
    words = []
    for s in segs:
        if getattr(s, "words", None):
            words += [(round(w.start, 2), round(w.end, 2), w.word) for w in s.words]
        else:
            words.append((round(s.start, 2), round(s.end, 2), s.text))
    return len(audio) / 16000, words, name


def whisper_for(row):
    cache = os.path.join(WH_DIR, row["book"], os.path.splitext(row["file"])[0] + ".json")
    if os.path.exists(cache):
        with open(cache, encoding="utf-8") as f:
            d = json.load(f)
        return d["duration"], [tuple(w) for w in d["words"]], d.get("model", ""), True
    src = row["audio"]
    if not src or not os.path.exists(src):
        raise FileNotFoundError(f"recording not found: {src or '(no path in the header)'}")
    local = local_copy(src)
    try:
        t0 = time.time()
        dur, words, name = run_whisper(local)
        dur = dur or probe_seconds(local)
        print(f"   Whisper heard {len(words)} words in {time.time() - t0:.0f}s", flush=True)
    finally:
        try:
            os.remove(local)
        except OSError:
            pass
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    tmp = cache + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"file": row["file"], "audio": src, "duration": dur, "model": name, "words": words},
                  f, ensure_ascii=False)
    os.replace(tmp, cache)
    return dur, words, name, False


# ------------------------------------------------------------------ one transcript
def _log(row, status, stats=None, gaps=(), note=""):
    stats = stats or {}
    new = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="", encoding="utf-8-sig" if new else "utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["time_pk", "book", "lesson", "file", "transcribed_by", "lines", "matched_pct",
                        "typical_shift_s", "biggest_shift_s", "end_before", "end_after", "gaps", "status", "note"])
        w.writerow([now_pk().strftime("%Y-%m-%d %I:%M %p"), row["book"], row["lesson"] if row["lesson"] != 9999 else "",
                    row["file"], row["model"], stats.get("eligible", ""),
                    round(100 * stats["coverage"]) if "coverage" in stats else "",
                    stats.get("median_shift", ""), stats.get("max_shift", ""),
                    fmt_hms(stats["end_old"]) if "end_old" in stats else "",
                    fmt_hms(stats["end_new"]) if "end_new" in stats else "",
                    " ".join(f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps), status, note])


def fix_one(row):
    with open(row["path"], encoding="utf-8") as f:
        text = f.read()
    stamp = (os.path.getmtime(row["path"]), len(text))
    if unfinished(text, row["check"]):
        return "skipped", "the transcription notebook will redo it"
    header, body = split_transcript(text)
    lines = parse_lines(body)
    if len(lines) < 15:
        return "skipped", f"only {len(lines)} timestamped lines found"
    dur, words, wmodel, cached = whisper_for(row)
    wt = WhisperText(words)
    new, kept, stats = align(lines, wt, dur)
    if new is None:
        _log(row, "not aligned", stats, note=stats.get("reason", ""))
        return "not aligned", stats.get("reason", "")
    gaps = find_gaps(lines, kept, wt, dur)
    notes = [f"- وقت: Whisper ({wmodel}) سے ملایا، {now_pk().date().isoformat()} · {round(100 * stats['coverage'])}% سطریں ملیں"
             f" · سب سے بڑی تبدیلی {fmt_hms(stats['max_shift'])} · اصل: transcripts/_original"]
    if gaps:
        notes.append("- خلا: " + " · ".join(f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps)
                     + " (یہاں آواز ہے مگر متن بہت کم — دوبارہ ٹرانسکرائب کرنا بہتر)")
    out = render(header, body, lines, new, notes)
    # the transcription notebook may have rewritten the file meanwhile: then leave it for the next run
    with open(row["path"], encoding="utf-8") as f:
        again = f.read()
    if (os.path.getmtime(row["path"]), len(again)) != stamp:
        return "skipped", "the transcript changed while we worked; next run"
    orig = os.path.join(ORIG_DIR, row["book"], row["file"])
    if not os.path.exists(orig):
        os.makedirs(os.path.dirname(orig), exist_ok=True)
        shutil.copyfile(row["path"], orig)
    tmp = row["path"] + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(out)
    os.replace(tmp, row["path"])
    status = "fixed, text missing" if gaps else "fixed"
    if gaps and MOVE_GAPS_FOR_REDO:
        dst = os.path.join(REDO_DIR, row["book"], row["file"])
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(row["path"], dst)
        status += ", moved for redo"
    _log(row, status, stats, gaps)
    return status, (f"{round(100 * stats['coverage'])}% matched · typical shift {stats['median_shift']:.0f}s · "
                    f"biggest {fmt_hms(stats['max_shift'])}" + (f" · text missing at " + ", ".join(
                        f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps) if gaps else ""))


# ------------------------------------------------------------------ report
def write_report():
    if not os.path.exists(LOG_PATH):
        return
    with open(LOG_PATH, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    last = {}
    for r in rows:                                     # latest result per transcript
        last[(r["book"], r["file"])] = r
    out = [f"# Timestamp repair — {now_pk().strftime('%d %b %Y, ')}{clock(now_pk())}", ""]
    fixed = [r for r in last.values() if r["status"].startswith("fixed")]
    gappy = [r for r in fixed if r["gaps"]]
    bad = [r for r in last.values() if r["status"] == "not aligned"]
    out += [f"- Fixed: {len(fixed)} transcripts ({len(gappy)} of them have stretches with text missing)",
            f"- Could not be matched: {len(bad)}", ""]
    if gappy:
        out += ["## Text missing (worth transcribing again)", "",
                "| کتاب | سبق | فائل | کہاں | ماڈل |", "|---|---|---|---|---|"]
        out += [f"| {r['book']} | {r['lesson']} | {r['file']} | {r['gaps']} | {r['transcribed_by']} |" for r in gappy]
        out.append("")
    if bad:
        out += ["## Could not be matched", ""] + [f"- {r['book']} · {r['file']}: {r['note']}" for r in bad] + [""]
    big = sorted(fixed, key=lambda r: -float(r["biggest_shift_s"] or 0))[:25]
    if big:
        out += ["## Biggest corrections", "", "| کتاب | فائل | ماڈل | سب سے بڑی تبدیلی | آخری وقت پہلے → اب |", "|---|---|---|---|---|"]
        out += [f"| {r['book']} | {r['file']} | {r['transcribed_by']} | {fmt_hms(float(r['biggest_shift_s'] or 0))} | "
                f"{r['end_before']} → {r['end_after']} |" for r in big]
    tmp = REPORT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write("\n".join(out) + "\n")
    os.replace(tmp, REPORT_PATH)


# ------------------------------------------------------------------ main loop
def run_fix():
    deadline = time.time() + MAX_HOURS * 3600
    rows = list_transcripts()
    todo = [r for r in rows if REFIX or not r["fixed"]]
    print(f"{len(rows)} transcripts in Drive, {len(todo)} to check"
          + (f" (books: {', '.join(ONLY_BOOKS)})" if ONLY_BOOKS else "") + ".")
    print(f"Stops by itself after {MAX_HOURS} h (at {clock(now_pk() + datetime.timedelta(hours=MAX_HOURS))}); "
          "stopping earlier is safe.\n", flush=True)
    done = {}
    try:
        for n, row in enumerate(todo, 1):
            if time.time() > deadline:
                print(f"\nTime limit reached (MAX_HOURS = {MAX_HOURS}). Run it again to carry on.")
                break
            print(f"[{n}/{len(todo)}] {row['book']} · {row['file']}", flush=True)
            try:
                status, note = fix_one(row)
            except Exception as e:
                status, note = "error", f"{type(e).__name__}: {str(e)[:160]}"
                _log(row, "error", note=note)
            done[status] = done.get(status, 0) + 1
            print(f"   {status}" + (f" — {note}" if note else ""), flush=True)
    except KeyboardInterrupt:
        print("\n■ Stopped. Finished transcripts are saved; run again to carry on.")
    write_report()
    print("\n" + ", ".join(f"{k}: {v}" for k, v in done.items()))
    print(f"Report: {REPORT_PATH}")

print('✓ ready')

In [ ]:
# @title 4 · Correct the timestamps (safe to stop and run again)
run_fix()

In [ ]:
# @title 5 · Check by ear (optional): open three corrected moments in Drive on a laptop
import random
ids = {}
try:
    with open(os.path.join(OUT_DIR, "recording_ids.csv"), encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            ids[r["file"]] = r["drive_id"]
except OSError:
    pass
fixed = [r for r in list_transcripts() if r["fixed"]]
if not fixed:
    print("Nothing corrected yet.")
else:
    row = random.choice(fixed)
    with open(row["path"], encoding="utf-8") as f:
        lines = parse_lines(split_transcript(f.read())[1])
    fid = ids.get(os.path.basename(row["audio"]), "")
    print(row["file"], "\n")
    for ln in random.sample(lines, min(3, len(lines))):
        link = f"https://drive.google.com/file/d/{fid}/view?t={ln['old']}" if fid else "(no Drive ID)"
        print(f"[{fmt_hms(ln['old'])}] {ln['raw'].strip()[:90]}\n   {link}\n")
    print("The teacher should be saying that sentence within a few seconds of where Drive opens.")

In [ ]:
# @title 6 · Light audio copies for the card pages (no GPU needed · safe to stop and run again)
# A small copy of each recording (AAC, 24 kbps, about 8 MB for 45 minutes) goes to My Drive/Wifaq 1448/web_audio/.
# Claude puts these copies inside the card pages, so a tap on a green time plays the teacher right there.
# Claude can fetch files up to 10 MB from Drive, so longer lessons are cut into equal parts.
WEB_BOOKS = ["ہدایہ جلد رابع"]   # books to prepare, in this order (add more later)
WEB_KBPS = 24
MAX_MB = 9.5
MAKE_WEB_AUDIO = True  # @param {type:"boolean"}

import math
WEB_DIR = os.path.join(OUT_DIR, "web_audio")


def _web_targets():
    with open(os.path.join(OUT_DIR, "recordings_inventory.csv"), encoding="utf-8-sig") as f:
        recs = list(csv.reader(f))[1:]
    out = []
    for rec in recs:
        rec = rec + [""] * (12 - len(rec))
        file, book, date, path = rec[0], rec[1], rec[10], rec[11]
        ranks = [i for i, b in enumerate(WEB_BOOKS) if _norm_name(b) and _norm_name(b) in _norm_name(book)]
        if path and ranks:
            out.append((ranks[0], date or "9999", file, book, path))
    return sorted(out)


def _encode(src, start, seconds, dst, kbps):
    tmp = os.path.join(LOCAL, "web_tmp.m4a")
    subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-ss", f"{start:.2f}", "-t", f"{seconds:.2f}", "-i", src,
                    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "aac", "-b:a", f"{kbps}k", "-movflags", "+faststart", tmp],
                   check=True, capture_output=True, timeout=3600)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(tmp, dst + ".part")
    os.replace(dst + ".part", dst)
    return os.path.getsize(dst)


def make_web_audio():
    targets = _web_targets()
    print(f"{len(targets)} recordings in {', '.join(WEB_BOOKS)}", flush=True)
    made = 0
    for n, (_, _, file, book, path) in enumerate(targets, 1):
        stem = os.path.splitext(file)[0]
        side = os.path.join(WEB_DIR, book, stem + ".json")
        if os.path.exists(side):
            continue
        try:
            local = local_copy(path)
            dur = probe_seconds(local)
            parts = max(1, math.ceil((dur * WEB_KBPS * 125 * 1.04 + 60000) / (MAX_MB * 1e6)))
            step = dur / parts
            info = []
            for i in range(parts):
                name = f"{stem}.m4a" if parts == 1 else f"{stem} - {i + 1}.m4a"
                dst = os.path.join(WEB_DIR, book, name)
                size = _encode(local, i * step, step if i < parts - 1 else dur - i * step + 1, dst, WEB_KBPS)
                if size > MAX_MB * 1e6:      # rare: very dense audio, so squeeze a little more
                    size = _encode(local, i * step, step if i < parts - 1 else dur - i * step + 1, dst, 16)
                info.append({"web_file": f"{book}/{name}", "start": round(i * step, 2), "size_mb": round(size / 1e6, 2)})
            with open(side, "w", encoding="utf-8") as f:
                json.dump({"file": file, "book": book, "seconds": round(dur, 1), "parts": info}, f, ensure_ascii=False)
            made += 1
            print(f"[{n}/{len(targets)}] {file}: {len(info)} part(s), {sum(p['size_mb'] for p in info):.1f} MB", flush=True)
        except Exception as e:
            print(f"[{n}/{len(targets)}] {file}: ✗ {type(e).__name__}: {str(e)[:150]}", flush=True)
        finally:
            try:
                os.remove(local)
            except Exception:
                pass
    # one index for Claude
    rows = []
    for side in glob.glob(os.path.join(WEB_DIR, "*", "*.json")):
        with open(side, encoding="utf-8") as f:
            d = json.load(f)
        for i, p in enumerate(d["parts"], 1):
            rows.append([d["book"], d["file"], i, p["start"], d["seconds"], p["size_mb"], p["web_file"]])
    with open(os.path.join(WEB_DIR, "index.csv"), "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["book", "file", "part", "start_sec", "lesson_seconds", "size_mb", "web_file"])
        w.writerows(sorted(rows))
    print(f"\n✓ {made} new, {len(rows)} files listed in web_audio/index.csv")


if MAKE_WEB_AUDIO:
    make_web_audio()
else:
    print("Skipped. Tick MAKE_WEB_AUDIO to make the light copies.")